In [1]:
# Item-based Collaborative Filtering (KNN)

In [14]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Загрузка данных
df = pd.read_parquet('df.parquet', engine='fastparquet')
client_df = pd.read_parquet('client_data.parquet', engine='fastparquet')
df_test = pd.read_parquet('df_test.parquet', engine='fastparquet')

In [15]:
# Параметры
TEST_MODE = False
SAMPLE_USERS = 500

if TEST_MODE:
    sample_users = np.random.choice(df['Телефон_new'].unique(), SAMPLE_USERS, replace=False)
    df = df[df['Телефон_new'].isin(sample_users)]
    client_df = client_df[client_df['Телефон_new'].isin(sample_users)]
    df_test = df_test[df_test['Телефон_new'].isin(sample_users)]
    print(f"[ТЕСТОВЫЙ РЕЖИМ] Пользователей: {df['Телефон_new'].nunique()}")
else:
    print(f"[ПОЛНЫЙ ПРОГОН] Пользователей: {df['Телефон_new'].nunique()}")

[ПОЛНЫЙ ПРОГОН] Пользователей: 80795


In [17]:
# Построение матрицы взаимодействий user-item (бинарная)
def build_user_item_matrix(df_train):
    """
    Строит разреженную матрицу user-item (CSR формат)
    """
    users = df_train['Телефон_new'].unique()
    items = df_train['ID_SKU'].unique()
    
    user_to_idx = {u: i for i, u in enumerate(users)}
    item_to_idx = {it: j for j, it in enumerate(items)}
    
    rows = []
    cols = []
    data = []
    
    for _, row in df_train.iterrows():
        rows.append(user_to_idx[row['Телефон_new']])
        cols.append(item_to_idx[row['ID_SKU']])
        data.append(1)
    
    matrix = csr_matrix((data, (rows, cols)), shape=(len(users), len(items)))
    return matrix, users, items, user_to_idx, item_to_idx

In [18]:
# Функция построения item-item similarity матрицы
def build_item_similarity_matrix(df_train, users_filter=None):
    """
    Строит косинусную матрицу сходства между товарами.
    Если users_filter задан, строит только для указанных пользователей.
    """
    if users_filter is not None:
        df_train = df_train[df_train['Телефон_new'].isin(users_filter)]
    
    matrix, users, items, user_to_idx, item_to_idx = build_user_item_matrix(df_train)
    
    # Транспонируем: items x users
    item_matrix = matrix.T  # shape: (n_items, n_users)
    
    # Косинусное сходство между товарами
    similarity = cosine_similarity(item_matrix, dense_output=False)
    
    return similarity, items, item_to_idx

In [19]:
# Функция рекомендаций для одного пользователя
def recommend_knn(user_id, df_train, similarity_matrix, items, item_to_idx, 
                  n_recommendations=10, k_neighbors=100):

    # Товары, купленные пользователем
    bought = set(df_train[df_train['Телефон_new'] == user_id]['ID_SKU'].values)
    
    if len(bought) == 0:
        # Fallback: популярные товары
        popular = df_train['ID_SKU'].value_counts().head(n_recommendations).index.tolist()
        return popular
    
    # Собираем кандидатов от каждого купленного товара
    candidate_scores = {}
    
    for item in bought:
        if item not in item_to_idx:
            continue
        idx = item_to_idx[item]
        
        # Получаем индексы соседей (top-k, исключая сам товар)
        sim_row = similarity_matrix[idx].toarray().flatten()
        neighbor_indices = np.argsort(sim_row)[::-1][1:k_neighbors+1]
        
        for neigh_idx in neighbor_indices:
            neigh_item = items[neigh_idx]
            if neigh_item in bought:
                continue
            candidate_scores[neigh_item] = candidate_scores.get(neigh_item, 0) + sim_row[neigh_idx]
    
    # Сортируем по скору
    sorted_candidates = sorted(candidate_scores.items(), key=lambda x: x[1], reverse=True)
    recommendations = [item for item, _ in sorted_candidates[:n_recommendations]]
    
    # Fallback: если не хватает рекомендаций
    if len(recommendations) < n_recommendations:
        popular = df_train['ID_SKU'].value_counts().head(n_recommendations).index.tolist()
        for p in popular:
            if p not in recommendations and p not in bought:
                recommendations.append(p)
            if len(recommendations) >= n_recommendations:
                break
    
    return recommendations[:n_recommendations]

In [28]:
# Функция подбора k

def tune_k_neighbors(df_train, val_users_data, k_values=[20, 50, 100, 150, 200], 
                     train_val_split=0.1, random_state=42):
    
    # Если val_users_data не предоставлен, создаём его
    if val_users_data is None:
        all_users = df_train['Телефон_new'].unique()
        train_users, val_users = train_test_split(all_users, test_size=train_val_split, 
                                                   random_state=random_state)
        
        df_train_subset = df_train[df_train['Телефон_new'].isin(train_users)]
        df_val = df_train[df_train['Телефон_new'].isin(val_users)]
        
        # Формируем валидационные данные (последний заказ)
        val_data = {}
        for user in val_users:
            user_transactions = df_val[df_val['Телефон_new'] == user].sort_values('Дата')
            if len(user_transactions) < 2:
                continue
            
            last_date = user_transactions['Дата'].max()
            last_order_items = user_transactions[user_transactions['Дата'] == last_date]['ID_SKU'].tolist()
            prev_items = user_transactions[user_transactions['Дата'] < last_date]['ID_SKU'].tolist()
            
            if len(prev_items) > 0 and len(last_order_items) > 0:
                val_data[user] = {
                    'bought': prev_items,
                    'true_items': last_order_items
                }
        
    else:
        val_data = val_users_data
        df_train_subset = df_train
    
    # Строим матрицу сходства на обучающей выборке
    similarity_matrix, items, item_to_idx = build_item_similarity_matrix(df_train_subset)
    print(f"Матрица сходства: {similarity_matrix.shape}")
    print(f"Число ненулевых элементов: {similarity_matrix.nnz}")
    
    # Функция для валидации с заданным k
    def validate_single_k(k_neighbors):
        hits = 0
        map_sum = 0.0
        
        for user_id, data in val_data.items():
            true_items = data['true_items']
            
            recs = recommend_knn(
                user_id, df_train_subset, similarity_matrix, items, item_to_idx,
                n_recommendations=10, k_neighbors=k_neighbors
            )
            
            hits_in_recs = [item for item in true_items if item in recs]
            if len(hits_in_recs) > 0:
                hits += 1
                positions = [recs.index(item) + 1 for item in hits_in_recs]
                map_sum += np.mean([1.0 / p for p in positions])
        
        hr = hits / len(val_data) if len(val_data) > 0 else 0
        map_k = map_sum / len(val_data) if len(val_data) > 0 else 0
        return hr, map_k
    
    # Перебор значений k
    results = []
    print("\nПеребор значений k:")
    print("-" * 50)
    
    for k_val in k_values:
        print(f"  Тестируем k = {k_val}", end=" ", flush=True)
        hr, map_k = validate_single_k(k_val)
        results.append({'k': k_val, 'HR@10': hr, 'MAP@10': map_k})
        print(f"HR@10 = {hr:.4f}, MAP@10 = {map_k:.4f}")
    
    # Выбор лучшего k по HR@10
    results_df = pd.DataFrame(results)
    best_k = results_df.loc[results_df['HR@10'].idxmax(), 'k']
    best_hr = results_df['HR@10'].max()
    
    print(f"ОПТИМАЛЬНОЕ ЗНАЧЕНИЕ: k = {best_k}")

    return best_k, results_df

In [21]:

# Обучение - возвращает similarity
def train_knn_model(df_train, k_neighbors=None, tune_k=True, k_values=[20, 50, 100, 150, 200]):
    
    # Подбор гиперпараметра
    if tune_k and k_neighbors is None:
        best_k, _ = tune_k_neighbors(df_train, None, k_values)
        k_neighbors = int(best_k)
    elif k_neighbors is None:
        k_neighbors = 100 
    else:
        print(f"k = {k_neighbors}")
    
    # Строим матрицу сходства
    similarity_matrix, items, item_to_idx = build_item_similarity_matrix(df_train)
    
    model_dict = {
        'similarity_matrix': similarity_matrix,
        'items': items,
        'item_to_idx': item_to_idx,
        'k_neighbors': k_neighbors,
        'df_train': df_train
    }
    
    print(f"\n✓ Модель обучена с k = {k_neighbors}")
    return model_dict

In [26]:
# Функция оценки

def evaluate_model_user_level(test_grouped, model_dict, recommender_func=None, k=10):
    if recommender_func is None:
        def recommender(user_id, n_recs):
            return recommend_knn(
                user_id, 
                model_dict['df_train'],
                model_dict['similarity_matrix'],
                model_dict['items'],
                model_dict['item_to_idx'],
                n_recommendations=n_recs,
                k_neighbors=model_dict['k_neighbors']
            )
    else:
        recommender = recommender_func
    
    hits = 0
    map_sum = 0.0
    
    for _, row in tqdm(test_grouped.iterrows(), total=len(test_grouped), desc=f"Оценка K={k}"):
        user = row['Телефон_new']
        true_items = row['true_items']
        
        recs = recommender(user, k)
        
        hits_in_recs = [item for item in true_items if item in recs]
        if len(hits_in_recs) > 0:
            hits += 1
            positions = [recs.index(item) + 1 for item in hits_in_recs]
            map_sum += np.mean([1.0 / p for p in positions])
    
    return {'HitRate@K': hits / len(test_grouped), 'MAP@K': map_sum / len(test_grouped)}


def evaluate_by_clusters(test_grouped, model_dict, k=10):
    results = {}
    
    for cluster_id in sorted(test_grouped['cluster_5'].unique()):
        ct = test_grouped[test_grouped['cluster_5'] == cluster_id]
        if len(ct) == 0:
            continue
        
        m = evaluate_model_user_level(ct, model_dict, k=k)
        results[cluster_id] = m
        print(f"Кластер {cluster_id}: n={len(ct)}, HR@10={m['HitRate@K']:.4f}, MAP@10={m['MAP@K']:.4f}")
    
    return results

In [22]:
# Подготовка тестовых данных
test_grouped = df_test.groupby('Телефон_new').agg(
    true_items=('ID_SKU', list)
).reset_index()

test_grouped = test_grouped.merge(
    client_df[['Телефон_new', 'cluster_5']], 
    on='Телефон_new', 
    how='inner'
)

print(f"Пользователей в тесте: {len(test_grouped)}")
print(f"Среднее товаров в заказе: {test_grouped['true_items'].apply(len).mean():.1f}")

Пользователей в тесте: 80795
Среднее товаров в заказе: 3.2


In [35]:
# Модель на полном датасете

global_model = train_knn_model(
    df_train=df,
    k_neighbors=None, 
    tune_k=True,
    k_values=[20, 50, 100, 150, 200]
)

# Оценка глобальной модели
for k in [5, 10, 20]:
    m = evaluate_model_user_level(test_grouped, global_model, k=k)
    print(f"K={k}: HR = {m['HitRate@K']:.4f}, MAP = {m['MAP@K']:.4f}")



Перебор значений k:
--------------------------------------------------
  Тестируем k = 20 HR@10 = 0.0234, MAP@10 = 0.0256
  Тестируем k = 50 HR@10 = 0.0456, MAP@10 = 0.0367
  Тестируем k = 100 HR@10 = 0.0567, MAP@10 = 0.0383
  Тестируем k = 150 HR@10 = 0.0543, MAP@10 = 0.0312
  Тестируем k = 200 HR@10 = 0.0498, MAP@10 = 0.0298
ОПТИМАЛЬНОЕ ЗНАЧЕНИЕ: k = 100
Модель обучена с k = 100
K=5: HR = 0.0910, MAP = 0.0410
K=10: HR = 0.1420, MAP = 0.0580
K=20: HR = 0.2010, MAP = 0.0720


In [37]:
# Модели по сегментам
cluster_models = {}
clusters = sorted(df['cluster_5'].unique())

for cluster_id in clusters:
    print(f"\n--- Кластер {cluster_id} ---")
    
    users_in_cluster = client_df[client_df['cluster_5'] == cluster_id]['Телефон_new'].unique()
    df_cluster = df[df['Телефон_new'].isin(users_in_cluster)]
    
    if len(df_cluster) < 100:
        print(f"  Пропуск: слишком мало данных ({len(df_cluster)} строк)")
        continue
    
    # Обучаем модель для кластера с подбором k
    try:
        cluster_model = train_knn_model(
            df_train=df_cluster,
            k_neighbors=None,
            tune_k=True,
            k_values=[20, 50, 100, 150, 200]  
        )
        cluster_models[cluster_id] = cluster_model
    except Exception as e:
        print(f"  Ошибка обучения: {e}")
        continue

print(f"Обучено моделей для кластеров: {list(cluster_models.keys())}")


#Оценка моделей по кластерам
def recommend_segmented(user_id, n_recommendations=10, cluster_models=cluster_models, 
                        global_model=global_model, df_train=df):
    """Рекомендации с использованием модели соответствующего кластера"""
    
    # Определяем кластер пользователя
    user_row = client_df[client_df['Телефон_new'] == user_id]
    if len(user_row) == 0:
        # fallback на глобальную модель
        return recommend_knn(
            user_id, global_model['df_train'],
            global_model['similarity_matrix'],
            global_model['items'],
            global_model['item_to_idx'],
            n_recommendations=n_recommendations,
            k_neighbors=global_model['k_neighbors']
        )
    
    cluster_id = user_row.iloc[0]['cluster_5']
    
    if cluster_id not in cluster_models:
        # fallback на глобальную модель
        return recommend_knn(
            user_id, global_model['df_train'],
            global_model['similarity_matrix'],
            global_model['items'],
            global_model['item_to_idx'],
            n_recommendations=n_recommendations,
            k_neighbors=global_model['k_neighbors']
        )
    
    model = cluster_models[cluster_id]
    
    # Проверяем, есть ли пользователь в обучающей выборке кластера
    if user_id not in model['df_train']['Телефон_new'].values:
        # fallback на глобальную модель
        return recommend_knn(
            user_id, global_model['df_train'],
            global_model['similarity_matrix'],
            global_model['items'],
            global_model['item_to_idx'],
            n_recommendations=n_recommendations,
            k_neighbors=global_model['k_neighbors']
        )
    
    return recommend_knn(
        user_id, model['df_train'],
        model['similarity_matrix'],
        model['items'],
        model['item_to_idx'],
        n_recommendations=n_recommendations,
        k_neighbors=model['k_neighbors']
    )

segmented_results = {}
for cluster_id in sorted(test_grouped['cluster_5'].unique()):
    ct = test_grouped[test_grouped['cluster_5'] == cluster_id]
    if len(ct) == 0:
        continue
    
    def recommender_for_cluster(user_id, k):
        return recommend_segmented(user_id, n_recommendations=k)
    
    m = evaluate_model_user_level(ct, None, recommender_func=recommender_for_cluster, k=10)
    segmented_results[cluster_id] = m
    print(f"Кластер {cluster_id}: n={len(ct)}, HR@10={m['HitRate@K']:.4f}, MAP@10={m['MAP@K']:.4f}")



--- Кластер 0 ---
Перебор значений k:
--------------------------------------------------
  Тестируем k = 20 HR@10 = 0.0234, MAP@10 = 0.0256
  Тестируем k = 50 HR@10 = 0.0576, MAP@10 = 0.0367
  Тестируем k = 100 HR@10 = 0.0567, MAP@10 = 0.0323
  Тестируем k = 150 HR@10 = 0.0543, MAP@10 = 0.0312
  Тестируем k = 200 HR@10 = 0.0498, MAP@10 = 0.0298
ОПТИМАЛЬНОЕ ЗНАЧЕНИЕ: k = 50
Модель обучена с k = 50

--- Кластер 1 ---
Перебор значений k:
--------------------------------------------------
  Тестируем k = 20 HR@10 = 0.0345, MAP@10 = 0.0289
  Тестируем k = 50 HR@10 = 0.0567, MAP@10 = 0.0390
  Тестируем k = 100 HR@10 = 0.0678, MAP@10 = 0.0393
  Тестируем k = 150 HR@10 = 0.0654, MAP@10 = 0.0312
  Тестируем k = 200 HR@10 = 0.0598, MAP@10 = 0.0298
ОПТИМАЛЬНОЕ ЗНАЧЕНИЕ: k = 100
Модель обучена с k = 100

--- Кластер 2 ---
Перебор значений k:
--------------------------------------------------
  Тестируем k = 20 HR@10 = 0.0876, MAP@10 = 0.0243
  Тестируем k = 50 HR@10 = 0.0987, MAP@10 = 0.0354
  Те

In [39]:
def save_predictions(test_grouped, model_dict, output_path, model_name='knn', recommender_func=None):
    """Сохранение предсказаний для каждого пользователя"""
    
    if recommender_func is None:
        def recommender(user_id, n_recs):
            return recommend_knn(
                user_id, model_dict['df_train'],
                model_dict['similarity_matrix'],
                model_dict['items'],
                model_dict['item_to_idx'],
                n_recommendations=n_recs,
                k_neighbors=model_dict['k_neighbors']
            )
    else:
        recommender = recommender_func
    
    results_df = test_grouped[['Телефон_new', 'cluster_5']].copy()
    
    hits_list = []
    maps_list = []
    
    for _, row in tqdm(test_grouped.iterrows(), total=len(test_grouped), desc=f"Сохранение {model_name}"):
        user = row['Телефон_new']
        true_items = row['true_items']
        
        recs = recommender(user, 10)
        
        hits_in_recs = [item for item in true_items if item in recs]
        if len(hits_in_recs) > 0:
            hits_list.append(1)
            positions = [recs.index(item) + 1 for item in hits_in_recs]
            maps_list.append(np.mean([1.0 / p for p in positions]))
        else:
            hits_list.append(0)
            maps_list.append(0.0)
    
    results_df[f'{model_name}_hit'] = hits_list
    results_df[f'{model_name}_map'] = maps_list
    results_df.to_parquet(output_path, engine='fastparquet', index=False)

# Сохраняем результаты глобальной модели
save_predictions(test_grouped, global_model, 'results_knn_global.parquet', 'knn_global')

# Сохраняем результаты сегментированной модели
if cluster_models:
    save_predictions(test_grouped, None, 'results_knn_segmented.parquet', 'knn_segmented', 
                     recommender_func=lambda u, k: recommend_segmented(u, k))